# Distributional arithmetic visualization notebook

This notebook walks through the `distarith` prototype with visual checks for the most important idea: random variables are expression graphs, not just marginal distributions. Reusing a source preserves dependence, while `iid()` creates a new independent source.

The examples use only the small prototype package plus `matplotlib` for charts.

In [ ]:
from distarith import Empirical, LogNormal, Normal, P, StudentT

try:
    import matplotlib.pyplot as plt
except ImportError as exc:
    raise RuntimeError(
        "This notebook needs matplotlib for visualization. Install it with `pip install matplotlib`."
    ) from exc

SEED = 7
SAMPLES = 20_000

## 1. Shared source versus independent copy

The expression `x - x` should collapse to zero because both sides reference the same source node. In contrast, `x - x.iid()` samples two independent source nodes with the same Normal marginal distribution.

In [ ]:
x = Normal(0, 1, name="x")
shared_difference = (x - x).sample(SAMPLES, seed=SEED)
independent_difference = (x - x.iid()).sample(SAMPLES, seed=SEED)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].hist(shared_difference, bins=30, color="#4c78a8")
axes[0].set_title("x - x: shared source")
axes[0].set_xlabel("value")
axes[0].set_ylabel("count")

axes[1].hist(independent_difference, bins=80, color="#f58518")
axes[1].set_title("x - x.iid(): independent sources")
axes[1].set_xlabel("value")

print("variance(x - x):", (x - x).variance(size=SAMPLES, seed=SEED))
print("variance(x - x.iid()):", (x - x.iid()).variance(size=SAMPLES, seed=SEED))

## 2. Profit distribution from random revenue and cost

Here, arithmetic builds a lazy random-variable expression. Sampling evaluates the graph jointly and produces particles for the derived `profit` distribution.

In [ ]:
revenue = LogNormal(mu=5.0, sigma=0.4, name="revenue")
cost = Normal(mu=120, sigma=15, name="cost")
profit = revenue - cost

profit_samples = profit.sample(SAMPLES, seed=SEED)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(profit_samples, bins=90, color="#54a24b", alpha=0.85)
ax.axvline(0, color="black", linestyle="--", linewidth=1.5, label="break even")
ax.axvline(profit.quantile(0.05, size=SAMPLES, seed=SEED), color="#e45756", linewidth=2, label="5% quantile")
ax.set_title("Particle approximation of profit = revenue - cost")
ax.set_xlabel("profit")
ax.set_ylabel("count")
ax.legend()

print("mean profit:", round(profit.mean(size=SAMPLES, seed=SEED), 2))
print("5% profit quantile:", round(profit.quantile(0.05, size=SAMPLES, seed=SEED), 2))
print("P(profit < 0):", round(P(profit < 0, size=SAMPLES, seed=SEED), 3))

## 3. Event visualization

Comparisons such as `profit < 0` create event objects. The probability helper evaluates the event with the same joint source samples used by the expression graph.

In [ ]:
losses = [sample for sample in profit_samples if sample < 0]
gains = [sample for sample in profit_samples if sample >= 0]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(losses, bins=45, color="#e45756", alpha=0.85, label="profit < 0")
ax.hist(gains, bins=70, color="#72b7b2", alpha=0.75, label="profit >= 0")
ax.axvline(0, color="black", linestyle="--", linewidth=1.5)
ax.set_title("Event split for profit particles")
ax.set_xlabel("profit")
ax.set_ylabel("count")
ax.legend()

print(f"Estimated loss probability: {len(losses) / len(profit_samples):.3f}")

## 4. Empirical inputs

The MVP can also use observed samples as a source distribution. This is useful when one part of a model comes from historical data and another part is parametric.

In [ ]:
historical_returns = Empirical([-0.05, -0.02, 0.00, 0.01, 0.03, 0.06, 0.08], name="returns")
fee = Normal(0.01, 0.002, name="fee")
net_return = historical_returns - fee

net_samples = net_return.sample(SAMPLES, seed=SEED)
quantiles = net_return.quantile([0.05, 0.25, 0.5, 0.75, 0.95], size=SAMPLES, seed=SEED)

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(net_samples, bins=60, color="#b279a2", alpha=0.85)
for q, value in zip([0.05, 0.25, 0.5, 0.75, 0.95], quantiles):
    ax.axvline(value, linewidth=1.4, label=f"q={q}: {value:.3f}")
ax.set_title("Net return from empirical returns minus random fee")
ax.set_xlabel("net return")
ax.set_ylabel("count")
ax.legend()

print("quantiles:", [round(value, 4) for value in quantiles])

## 5. Fat-tail shock added to normal noise

A Student-t source with low degrees of freedom is a simple fat-tail distribution. Adding it to a Normal source keeps the center familiar, but makes extreme outcomes much more common. This is useful for studying rare-loss or stress scenarios.

In [ ]:
normal_noise = Normal(0, 1, name="normal_noise")
fat_tail_shock = StudentT(df=3, loc=0, scale=1, name="fat_tail_shock")
normal_plus_fat_tail = normal_noise + fat_tail_shock

normal_samples = normal_noise.sample(SAMPLES, seed=SEED)
fat_tail_samples = fat_tail_shock.sample(SAMPLES, seed=SEED)
combined_samples = normal_plus_fat_tail.sample(SAMPLES, seed=SEED)

threshold = 4
normal_tail_probability = sum(abs(sample) > threshold for sample in normal_samples) / SAMPLES
combined_tail_probability = sum(abs(sample) > threshold for sample in combined_samples) / SAMPLES

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
axes[0].hist(normal_samples, bins=90, color="#4c78a8", alpha=0.85)
axes[0].axvline(-threshold, color="black", linestyle="--", linewidth=1.2)
axes[0].axvline(threshold, color="black", linestyle="--", linewidth=1.2)
axes[0].set_title("Normal noise")
axes[0].set_xlabel("value")
axes[0].set_ylabel("count")

axes[1].hist(combined_samples, bins=130, color="#f58518", alpha=0.85)
axes[1].axvline(-threshold, color="black", linestyle="--", linewidth=1.2)
axes[1].axvline(threshold, color="black", linestyle="--", linewidth=1.2)
axes[1].set_title("Normal + Student-t(df=3) fat-tail shock")
axes[1].set_xlabel("value")

print(f"P(|Normal| > {threshold}): {normal_tail_probability:.4f}")
print(f"P(|Normal + fat-tail shock| > {threshold}): {combined_tail_probability:.4f}")
print("Normal + fat-tail 1%, 5%, 50%, 95%, 99% quantiles:")
print([round(value, 3) for value in normal_plus_fat_tail.quantile([0.01, 0.05, 0.5, 0.95, 0.99], size=SAMPLES, seed=SEED)])

## 6. What to look for

- The first chart should show a spike at exactly zero for `x - x`, proving source identity is preserved.
- The independent-copy chart should spread out with variance near 2.
- The profit chart shows how an expression returns a distribution-like particle approximation instead of a single scalar.
- The event chart makes `P(profit < 0)` visible as the red mass left of zero.
- The fat-tail chart shows how adding a Student-t shock to Normal noise increases the probability of extreme outcomes beyond the threshold lines.

Future notebook iterations can add planner diagnostics, symbolic simplification examples, and FFT/grid approximations as those capabilities are implemented.